# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/muzammilsharf/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

- Unit of analysis: One row represents the daily search and web performance for a single specific piece of content (content_hash_id) belonging to a specific client (client_hash_id).

- Time window: The entire month of March 2026 (report_date spanning 2026-03-01 to 2026-03-31).

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

- Feature: gsc_clicks, gsc_impressions, ga4_sessions, sessions_ai, gsc_avg_position. (Knowable at the decision moment because daily logs are finalized and sealed at midnight).

- Label: is_declining_label. (A proxy target predicting whether gsc_clicks drop significantly in the subsequent 30-day window).

- Context: report_date, client_hash_id, content_hash_id. (Pure identifiers, never passed to the ML model).

- Excluded: Any data from April 2026 onward. Why: Allowing the model to see future performance metrics while training on March data would cause target leakage, giving it the "answer key" before it makes a prediction.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [2]:
from huggingface_hub import list_repo_files
import getpass

# 1. This will prompt you to paste your token securely
print("Paste your Hugging Face READ token below and press Enter:")
hf_token = getpass.getpass()

# 2. List all files in the gated repo using your token
print("\nFetching file list...")
all_files = list_repo_files("FlyRank/internship-warehouse", repo_type="dataset", token=hf_token)

# 3. Filter down to just the daily performance table files
daily_files = [f for f in all_files if "fact_content_daily_performance" in f]

# 4. Print them out so we can find the March 2026 one
for f in daily_files:
    print(f)

Paste your Hugging Face READ token below and press Enter:

Fetching file list...
fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/month=2025-02/data_0.parquet
fact_content_daily_performance/month=2025-03/data_0.parquet
fact_content_daily_performance/month=2025-04/data_0.parquet
fact_content_daily_performance/month=2025-05/data_0.parquet
fact_content_daily_performance/month=2025-06/data_0.parquet
fact_content_daily_performance/month=2025-07/data_0.parquet
fact_content_daily_performance/month=2025-08/data_0.parquet
fact_content_daily_performance/month=2025-09/data_0.parquet
fact_content_daily_performance/month=2025-10/data_0.parquet
fact_content_daily_performance/month=2025-11/data_0.parquet
fact_content_daily_performance/month=2025-12/data_0.parquet
fact_content_daily_performance/month=2026-01/data_0.parquet
fact_content_daily_performance/month=2026-02/data_0.parquet
fact_content_daily_performance/month=2026-03/data_0.parquet
fact_content_daily_

In [3]:
import pandas as pd
from datasets import load_dataset

# The exact path to the March 2026 data
march_file = "fact_content_daily_performance/month=2026-03/data_0.parquet"

# Load just this specific file
print("Downloading March 2026 data...")
dataset = load_dataset(
    "FlyRank/internship-warehouse", 
    data_files=march_file, 
    split="train",
    token=hf_token
)

# Convert it to the Pandas DataFrame we've been trying to make
df = dataset.to_pandas()

# Show the columns and the first 5 rows
print("\nColumns:", df.columns.tolist())
df.head()

Generating train split: 9841378 examples [01:28, 111117.51 examples/s]



Columns: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events']


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,None,20,0,67,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,None,1,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,None,125,1,616,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,None,7,0,28,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,None,11,0,25,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
# 1. Verify the Grain: Should return 0 if one row = one unique piece of content per client, per day
grain_cols = ['client_hash_id', 'content_hash_id', 'report_date']
grain_violations = df.duplicated(subset=grain_cols).sum()
print(f"1. Grain violations (duplicates): {grain_violations}")

# 2. Verify the Time Window & Counts
print(f"2. Scope: {len(df):,} rows")
print(f"   Date span: {df['report_date'].min()} to {df['report_date'].max()}")

# 3. Verify Availability (Filtering with IS TRUE as required)
# Let's see how many rows actually have Google Search Console data available
gsc_survivors = df[df['gsc_data_available'] == True]
print(f"3. Availability: {len(gsc_survivors):,} rows survive the 'gsc_data_available IS TRUE' filter")

1. Grain violations (duplicates): 0
2. Scope: 9,841,378 rows
   Date span: 2026-03-01 to 2026-03-31
3. Availability: 3,611,061 rows survive the 'gsc_data_available IS TRUE' filter


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

- Algorithmic Volatility: This data shows historical Google Search Console and GA4 metrics, but cannot predict sudden search engine core algorithm updates that might instantly tank a previously healthy page.

- Missing Context: I have engagement metrics (like ga4_total_engagement_sec), but this slice doesn't contain the actual text or quality of the content. A page might decline simply because the information on it became outdated, which numeric logs alone cannot flag.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.